## Simulator (fleet of uavs)

In [1]:
from simulator import Simulator
from simulator.config import Color
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.helpers.processes import create_process
from simulator.planner import AutoPlan, Plan
from simulator.visualizer import (
    QGC,
    Gazebo,
    GazMarker,
    NoVisualizer,
    QGCMarker,
    SimVehicle,
)

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241,alt=0,heading=0) 
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading) 

base_home = ENUPose(x=5, y=0, z=0, heading=0)
base_intruder_center = ENU(x=0,y=50,z=5) #15,15

enu_home = enu_origin.to_abs(base_home)
gra_home = gra_origin.to_abs(base_home)

enu_intruder_center = enu_home.to_abs(base_intruder_center).unpose()
gra_intruder_center = gra_home.to_abs(base_intruder_center).unpose()
intruder_radius = 7 #5

## Create Vehicles

In [3]:
sysid = 1
color = Color.BLUE
    
side_len = 100
alt = 6


auto_plan = AutoPlan.square_traj(
    side_len=side_len,
    alt=alt,
    name="simple_auto_plan",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=base_home,
)


veh = SimVehicle.from_relative(
    sysid=sysid,
    gcs_name=f'Multicolor_{color.emoji}',
    plan=auto_plan,
    color=color,
    enu_origin=enu_origin,
    relative_home=base_home,
    relative_path=Plan.create_square_path(side_len=side_len, alt=alt),
    model="iris",
)

## Visualizer

### Gazebo

In [4]:
gaz= Gazebo(gra_origin,world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_gaz = GazMarker(name="origin",
                    group="origin",
                    pos=enu_origin.unpose(),
                    color=Color.WHITE)
gaz.markers.append(origin_gaz)

intruder_sphere_gaz = GazMarker(name="intruder_sphere",
                    group="intruder_sphere",
                    pos=enu_intruder_center,
                    radius=intruder_radius,
                    color=Color.RED,
                    alpha=0.9)
intruder_center_gaz = GazMarker(name="intruder_center",
                    group="intruder_center",
                    pos=enu_intruder_center,
                    color=Color.RED,
                    )
gaz.markers.append(intruder_sphere_gaz)
gaz.markers.append(intruder_center_gaz)

### QGroundControl

In [5]:
qgc= QGC(gra_origin)
origin_qgc = QGCMarker(name="origin",
                pos=gra_origin.unpose(),
                color=Color.WHITE
                )
qgc.markers.append(origin_qgc)

### No Visualizer

In [6]:
novis = NoVisualizer(gra_origin)

## Simulator

In [7]:
simulator = Simulator(
	visualizer=gaz,
	terminals=['gcs', 'adsb_socat', 'adsb_injector'],
	verbose=1,
)

simulator.add_vehicle(veh)

simulator.show()

## Launch avoidance processes

In [8]:
gra_intruder_center

GRA(lat=-35.362877335678725, lon=149.16527911589873, alt=5.000198598249823)

## Run

In [ ]:
orac = simulator.launch()
orac.run()

11:46:44 - Oracle ⚪ - INFO - 🖥️  Gazebo launched for realistic simulation and 3D visualization.
11:46:44 - Oracle ⚪ - INFO - 🚀 GCS Multicolor_🟦 launched (PID 3320454)
11:46:44 - Oracle ⚪ - INFO - 🏁 Starting Oracle with 1 vehicles and 1 GCSs
